# Notebook 3 — PaperLens Advanced AI Research Assistant

**Project:** Research Paper Intelligence Assistant / PaperLens  
**Runtime:** Google Colab (GPU recommended for optional transformer models)  
**Level:** Beginner-friendly, with comments and modular functions

This notebook upgrades the first two notebooks with a practical, modern research pipeline:

- **Semantic search** using Sentence Transformers (`all-MiniLM-L6-v2`)
- **Hybrid retrieval** with dense embeddings + BM25 + TF-IDF, fused using Reciprocal Rank Fusion (RRF)
- **Optional cross-encoder reranking** for more precise top results
- **Evidence-grounded extractive question answering** with source paper/page citations
- **Extractive and optional transformer-generated summaries**
- **Keyphrase extraction**, topic discovery, document similarity, paper comparison, DOI/reference mining
- **Optional OCR** for scanned PDFs and **optional table extraction**
- **Retrieval evaluation** scaffold (metrics are only meaningful after you annotate relevant passages)
- **Downloadable CSV/JSON/ZIP exports** for analysis and website integration

> **Trust and privacy:** this notebook processes PDFs you upload to your Colab runtime. Embedding and optional transformer models may be downloaded from Hugging Face. Do not upload confidential papers unless you are comfortable with your Colab environment. Generated summaries/answers can be wrong; always inspect the cited source pages. No benchmark score is invented by this notebook.

## 0. How to run this notebook

1. Open the notebook in [Google Colab](https://colab.research.google.com/) and choose **Runtime → Change runtime type → T4 GPU** if available. The core workflow can run on CPU, but embedding and optional transformer tasks are faster on a GPU.
2. Run cells from top to bottom.
3. Upload two or more text-based PDFs when prompted. One PDF is enough to test retrieval; multiple PDFs unlock comparison.
4. Optional heavy features are in separate cells and are **not required** for the core search pipeline.
5. At the end, download the `paperlens_advanced_exports.zip` archive. The default website export contains metadata and insights, not the full extracted text.

## 1. Install core libraries

This cell installs the packages needed for semantic embeddings, lexical retrieval, analysis, PDF extraction, and visualization. Optional OCR/table/LLM dependencies are installed only in their own sections.

In [ ]:
%pip -q install pymupdf sentence-transformers rank-bm25 scikit-learn pandas numpy matplotlib

## 2. Import libraries and choose device

In [ ]:
import os
import re
import json
import math
import shutil
import zipfile
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from google.colab import files

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import NMF
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print(f"Environment ready. Device: {DEVICE}")
print("Core libraries imported successfully.")

## 3. Upload research PDFs

Upload the PDFs you are allowed to process. Papers are kept in this runtime session; the notebook does not automatically publish them.

In [ ]:
uploaded_files = files.upload()
if not uploaded_files:
    raise ValueError("No PDFs were uploaded. Re-run this cell and choose at least one PDF.")

pdf_files = {
    filename: content
    for filename, content in uploaded_files.items()
    if filename.lower().endswith(".pdf")
}
if not pdf_files:
    raise ValueError("No .pdf file was selected. Please upload one or more PDF files.")

print(f"Loaded {len(pdf_files)} PDF file(s):")
for filename in pdf_files:
    print(" •", filename)

## 4. Extract page text and document metadata

In [ ]:
def clean_page_text(text: str) -> str:
    """Normalize extracted PDF text without removing scientific symbols unnecessarily."""
    text = text.replace("\x00", " ")
    text = re.sub(r"-\s*\n\s*(?=[A-Za-z])", "", text)  # join line-break hyphenation
    text = re.sub(r"[\t\r]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

page_records = []
paper_metadata = []
extraction_errors = []

for filename, pdf_bytes in pdf_files.items():
    try:
        with fitz.open(stream=pdf_bytes, filetype="pdf") as pdf:
            metadata = pdf.metadata or {}
            paper_metadata.append({
                "paper": filename,
                "title": (metadata.get("title") or "").strip() or Path(filename).stem,
                "author": (metadata.get("author") or "").strip(),
                "subject": (metadata.get("subject") or "").strip(),
                "page_count": len(pdf),
                "pdf_creation_date": (metadata.get("creationDate") or "").strip(),
            })
            for page_index, page in enumerate(pdf, start=1):
                text = clean_page_text(page.get_text("text"))
                page_records.append({"paper": filename, "page": page_index, "text": text,
                                     "word_count": len(text.split())})
    except Exception as exc:
        extraction_errors.append({"paper": filename, "error": str(exc)})

pages_df = pd.DataFrame(page_records)
papers_df = pd.DataFrame(paper_metadata)
if pages_df.empty:
    raise ValueError("No PDF pages could be read. Check whether the files are valid PDFs.")

print(f"Extracted {len(pages_df):,} pages across {len(papers_df):,} paper(s).")
display(papers_df)
if extraction_errors:
    print("Files that could not be read:")
    display(pd.DataFrame(extraction_errors))
if not pages_df["text"].str.len().gt(0).any():
    print("Warning: no selectable text was found. The PDFs may be scanned; see the optional OCR section.")

## 5. Build overlapping, page-aware chunks

Chunking keeps each passage connected to its original paper and 1-based PDF page. Small overlaps reduce the chance that an important statement is cut at a chunk boundary.

In [ ]:
def make_chunks(page_table: pd.DataFrame, chunk_words: int = 220, overlap_words: int = 45):
    if chunk_words <= overlap_words:
        raise ValueError("chunk_words must be larger than overlap_words.")
    records = []
    step = chunk_words - overlap_words
    for row in page_table.itertuples(index=False):
        words = str(row.text).split()
        for start in range(0, len(words), step):
            piece = " ".join(words[start:start + chunk_words]).strip()
            if len(piece) < 80:
                continue
            records.append({
                "chunk_id": f"{row.paper}::p{row.page}::w{start}",
                "paper": row.paper,
                "page": int(row.page),
                "start_word": start,
                "text": piece,
            })
    return pd.DataFrame(records)

chunks_df = make_chunks(pages_df)
if chunks_df.empty:
    raise ValueError("No usable text chunks were created. If your PDFs are scanned, go to the optional OCR section, run OCR for blank pages, then re-run this chunking cell.")
print(f"Created {len(chunks_df):,} chunks.")
display(chunks_df[["chunk_id", "paper", "page", "text"]].head(3))

## 6. Build three retrieval indexes

- **Dense semantic retrieval**: can find related concepts even if the exact words differ.
- **BM25**: strong lexical retrieval for exact technical terms.
- **TF-IDF**: a transparent baseline used for comparison.
- **Hybrid RRF**: merges ranked lists so no single signal dominates. Scores are ranking signals, not probabilities.

In [ ]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
chunk_texts = chunks_df["text"].astype(str).tolist()
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

TOKEN_RE = re.compile(r"[A-Za-z0-9][A-Za-z0-9_'-]*")
def lexical_tokens(text: str):
    return [token.lower() for token in TOKEN_RE.findall(str(text))]

bm25_corpus = [lexical_tokens(text) for text in chunk_texts]
bm25_index = BM25Okapi(bm25_corpus)
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True, stop_words="english", ngram_range=(1, 2),
    max_features=150_000, sublinear_tf=True
)
tfidf_matrix = tfidf_vectorizer.fit_transform(chunk_texts)

print("Indexes created:")
print(f" • Semantic vectors: {chunk_embeddings.shape}")
print(f" • BM25 chunks: {len(bm25_corpus):,}")
print(f" • TF-IDF matrix: {tfidf_matrix.shape}")

## 7. Hybrid search, result fusion, and citations

In [ ]:
def _rank_positions(scores: np.ndarray) -> np.ndarray:
    """Return 1-based rank for every item (1 means best)."""
    order = np.argsort(-np.asarray(scores))
    ranks = np.empty(len(order), dtype=int)
    ranks[order] = np.arange(1, len(order) + 1)
    return ranks


def search_papers(query: str, top_k: int = 8, mode: str = "hybrid") -> pd.DataFrame:
    """Search chunks using semantic, BM25, TF-IDF, or RRF-fused hybrid ranking."""
    query = str(query).strip()
    if not query:
        raise ValueError("Enter a non-empty search query.")
    if mode not in {"semantic", "bm25", "tfidf", "hybrid"}:
        raise ValueError("mode must be semantic, bm25, tfidf, or hybrid")

    query_embedding = embedding_model.encode([query], convert_to_numpy=True,
                                             normalize_embeddings=True)[0]
    semantic_scores = chunk_embeddings @ query_embedding
    bm25_scores = np.asarray(bm25_index.get_scores(lexical_tokens(query)), dtype=float)
    query_tfidf = tfidf_vectorizer.transform([query])
    tfidf_scores = cosine_similarity(query_tfidf, tfidf_matrix).ravel()

    semantic_ranks = _rank_positions(semantic_scores)
    bm25_ranks = _rank_positions(bm25_scores)
    tfidf_ranks = _rank_positions(tfidf_scores)
    # Weighted Reciprocal Rank Fusion: stable when retrieval methods have different score scales.
    rrf_scores = (0.50 / (60 + semantic_ranks) +
                  0.30 / (60 + bm25_ranks) +
                  0.20 / (60 + tfidf_ranks))

    frame = chunks_df.copy()
    frame["semantic_score"] = semantic_scores
    frame["bm25_score"] = bm25_scores
    frame["tfidf_score"] = tfidf_scores
    frame["hybrid_score"] = rrf_scores
    score_column = {"semantic": "semantic_score", "bm25": "bm25_score",
                    "tfidf": "tfidf_score", "hybrid": "hybrid_score"}[mode]
    frame["ranking_method"] = mode
    frame["score"] = frame[score_column]
    return frame.sort_values("score", ascending=False).head(max(1, int(top_k))).reset_index(drop=True)


def show_search_results(results: pd.DataFrame, excerpt_chars: int = 850):
    if results.empty:
        print("No matching passages found.")
        return
    for rank, row in enumerate(results.itertuples(index=False), start=1):
        display(Markdown(
            f"### {rank}. {row.paper} — page {row.page}\n"
            f"**Chunk ID:** `{row.chunk_id}` · **{row.ranking_method} score:** `{row.score:.4f}`\n\n"
            f"> {row.text[:excerpt_chars]}{'…' if len(row.text) > excerpt_chars else ''}"
        ))

example_query = "What methods, datasets, and limitations are discussed?"
example_results = search_papers(example_query, top_k=5, mode="hybrid")
show_search_results(example_results)

## 8. Optional: Cross-encoder reranking

A bi-encoder retrieves candidates quickly; a cross-encoder reads each query–passage pair together and can improve top-result ordering. This downloads another model and is therefore optional.

In [ ]:
# Run this cell only if you want the extra reranker model downloaded.
# On a free Colab runtime, start with the hybrid results above if memory is limited.

CROSS_ENCODER = None

def rerank_with_cross_encoder(query: str, candidates: pd.DataFrame, top_k: int = 8) -> pd.DataFrame:
    global CROSS_ENCODER
    if candidates.empty:
        return candidates.copy()
    from sentence_transformers import CrossEncoder
    if CROSS_ENCODER is None:
        CROSS_ENCODER = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)
    pairs = [[str(query), str(text)] for text in candidates["text"].tolist()]
    scores = np.asarray(CROSS_ENCODER.predict(pairs, show_progress_bar=False), dtype=float)
    reranked = candidates.copy()
    reranked["reranker_score"] = scores
    return reranked.sort_values("reranker_score", ascending=False).head(top_k).reset_index(drop=True)

# Example, uncomment when ready:
# reranked_results = rerank_with_cross_encoder(example_query, example_results, top_k=5)
# show_search_results(reranked_results)

## 9. Extractive summaries with page-level references

In [ ]:
SENTENCE_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9(\"'])")

def sentence_records_for_paper(paper_name: str):
    subset = pages_df[pages_df["paper"] == paper_name]
    records = []
    for row in subset.itertuples(index=False):
        for sentence in SENTENCE_RE.split(str(row.text)):
            sentence = sentence.strip()
            if len(sentence.split()) >= 7 and len(sentence) >= 45:
                records.append({"paper": paper_name, "page": int(row.page), "sentence": sentence, "sentence_index": len(records)})
    return records


def extractive_summary(paper_name: str, n_sentences: int = 7) -> pd.DataFrame:
    records = sentence_records_for_paper(paper_name)
    if not records:
        return pd.DataFrame(columns=["paper", "page", "sentence", "score"])
    texts = [item["sentence"] for item in records]
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
    matrix = vectorizer.fit_transform(texts)
    scores = np.asarray(matrix.sum(axis=1)).ravel() / np.sqrt(np.maximum(1, np.asarray(matrix.getnnz(axis=1)).ravel()))
    for row, score in zip(records, scores):
        row["score"] = float(score)
    selected = sorted(sorted(records, key=lambda item: item["score"], reverse=True)[:max(1, int(n_sentences))],
                      key=lambda item: item["sentence_index"])
    return pd.DataFrame(selected)

paper_names = papers_df["paper"].tolist()
summary_paper = paper_names[0]
summary_df = extractive_summary(summary_paper, n_sentences=7)
print(f"Extractive summary: {summary_paper}")
for item in summary_df.itertuples(index=False):
    print(f"[p. {item.page}] {item.sentence}")

## 10. Evidence-grounded question answering (extractive)

This optional feature retrieves relevant passages first, then uses an extractive question-answering model to select a span from each retrieved passage. The outputs retain paper names, PDF page numbers, and verbatim evidence. The model may still choose a misleading span, so inspect its evidence and score.

In [ ]:
QA_PIPELINE = None

def load_extractive_qa():
    global QA_PIPELINE
    if QA_PIPELINE is None:
        from transformers import pipeline
        QA_PIPELINE = pipeline(
            "question-answering",
            model="deepset/roberta-base-squad2",
            device=0 if DEVICE == "cuda" else -1,
        )
    return QA_PIPELINE


def answer_question_extractive(question: str, retrieval_top_k: int = 6) -> pd.DataFrame:
    qa = load_extractive_qa()
    candidates = search_papers(question, top_k=retrieval_top_k, mode="hybrid")
    answers = []
    for row in candidates.itertuples(index=False):
        try:
            result = qa(question=question, context=str(row.text))
            answer = str(result.get("answer", "")).strip()
            score = float(result.get("score", 0.0))
            if answer and answer.lower() not in {"[cls]", "[sep]"}:
                answers.append({
                    "answer": answer,
                    "qa_score": score,
                    "paper": row.paper,
                    "page": int(row.page),
                    "chunk_id": row.chunk_id,
                    "evidence": row.text,
                })
        except Exception as exc:
            print(f"Skipped one candidate due to QA error: {exc}")
    if not answers:
        return pd.DataFrame(columns=["answer", "qa_score", "paper", "page", "chunk_id", "evidence"])
    return pd.DataFrame(answers).sort_values("qa_score", ascending=False).drop_duplicates("answer").head(5).reset_index(drop=True)

# This downloads a QA model on its first run. Uncomment to test with your own question.
# qa_results = answer_question_extractive("What are the main limitations?", retrieval_top_k=6)
# display(qa_results[["answer", "qa_score", "paper", "page", "evidence"]])

## 11. Optional: Generative summary with FLAN-T5

This is a heavier, experimental text-generation option. It asks the model to use only retrieved evidence and output source tags, but prompting is not a guarantee against hallucination. Confirm every claim against the cited PDF pages. The core notebook does not require this model.

In [ ]:
FLAN_GENERATOR = None

def load_flan_generator():
    global FLAN_GENERATOR
    if FLAN_GENERATOR is None:
        from transformers import pipeline
        FLAN_GENERATOR = pipeline(
            "text2text-generation",
            model="google/flan-t5-base",
            device=0 if DEVICE == "cuda" else -1,
        )
    return FLAN_GENERATOR


def grounded_answer_with_flan(question: str, top_k: int = 4) -> dict:
    results = search_papers(question, top_k=top_k, mode="hybrid")
    sources = []
    for i, row in enumerate(results.itertuples(index=False), start=1):
        # Keep the prompt small enough for a typical T5 input window.
        sources.append(f"[S{i}] Paper: {row.paper}; PDF page: {row.page}\nEvidence: {row.text[:900]}")
    context = "\n\n".join(sources)
    prompt = (
        "Answer the question using ONLY the evidence below. If the evidence does not answer it, "
        "say that the papers do not provide enough evidence. Do not add outside facts. "
        "Add source tags such as [S1] to each key claim.\n\n"
        f"QUESTION: {question}\n\nEVIDENCE:\n{context}\n\nANSWER:"
    )
    generator = load_flan_generator()
    answer = generator(prompt, max_new_tokens=180, do_sample=False, truncation=True)[0]["generated_text"]
    source_table = results[["paper", "page", "chunk_id"]].copy()
    source_table.insert(0, "source_tag", [f"S{i}" for i in range(1, len(source_table) + 1)])
    return {"answer": answer, "sources": source_table}

# Example — uncomment if you want to download FLAN-T5-base:
# generated = grounded_answer_with_flan("What limitations do the papers report?")
# display(Markdown(generated["answer"]))
# display(generated["sources"])

## 12. Keyphrase extraction (lightweight by default; KeyBERT optional)

In [ ]:
STOPWORDS_EXTRA = {
    "paper", "study", "result", "results", "method", "methods", "research", "based",
    "using", "used", "use", "approach", "proposed", "show", "shown", "also", "however",
    "therefore", "could", "would", "may", "might", "one", "two", "three", "figure", "table"
}

def simple_keyphrases(text: str, top_n: int = 15):
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 3), max_features=50_000,
                                 token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_-]+\b")
    matrix = vectorizer.fit_transform([text])
    weights = np.asarray(matrix.sum(axis=0)).ravel()
    features = np.asarray(vectorizer.get_feature_names_out())
    ranked = sorted(zip(features, weights), key=lambda item: item[1], reverse=True)
    return [(phrase, float(score)) for phrase, score in ranked
            if not any(word in STOPWORDS_EXTRA for word in phrase.split())][:top_n]

paper_keyword_map = {}
for paper in paper_names:
    paper_text = " ".join(pages_df.loc[pages_df.paper == paper, "text"].astype(str))
    paper_keyword_map[paper] = simple_keyphrases(paper_text, top_n=12)

for paper, phrases in paper_keyword_map.items():
    print("\n", paper)
    print(", ".join(phrase for phrase, _ in phrases[:10]))

### Optional KeyBERT semantic keyphrases

KeyBERT ranks phrases by how well their embeddings represent the document. This adds another model-based extraction technique; it is optional because KeyBERT has extra installation and runtime overhead.

In [ ]:
# Optional. Run if you want semantic keyphrases instead of the lightweight TF-IDF phrases.
# %pip -q install keybert
# from keybert import KeyBERT
# keybert_model = KeyBERT(model=embedding_model)
# paper_text = " ".join(pages_df.loc[pages_df.paper == paper_names[0], "text"].astype(str))
# keybert_model.extract_keywords(paper_text, keyphrase_ngram_range=(1, 3), top_n=12,
#                                stop_words="english", use_mmr=True, diversity=0.35)

## 13. Topic discovery with NMF

NMF discovers groups of terms that co-occur in chunks. The topics are exploratory labels, not verified scientific categories.

In [ ]:
def discover_topics(n_topics: int = 5, top_terms: int = 8):
    if len(chunk_texts) < 2:
        print("Topic discovery needs at least two text chunks.")
        return pd.DataFrame(), pd.DataFrame()
    vectorizer = TfidfVectorizer(stop_words="english", max_features=20_000,
                                 ngram_range=(1, 2), min_df=1)
    X = vectorizer.fit_transform(chunk_texts)
    count = min(max(2, int(n_topics)), X.shape[0], X.shape[1])
    if count < 2:
        print("Not enough distinct terms/chunks to discover multiple topics.")
        return pd.DataFrame(), pd.DataFrame()
    model = NMF(n_components=count, random_state=SEED, init="nndsvda", max_iter=400)
    W = model.fit_transform(X)
    H = model.components_
    feature_names = np.asarray(vectorizer.get_feature_names_out())
    topic_rows = []
    for topic_id, weights in enumerate(H, start=1):
        top_ids = np.argsort(weights)[::-1][:top_terms]
        topic_rows.append({"topic_id": topic_id,
                           "top_terms": ", ".join(feature_names[top_ids]),
                           "chunk_count": int((W.argmax(axis=1) == topic_id - 1).sum())})
    assignments = chunks_df[["chunk_id", "paper", "page"]].copy()
    assignments["topic_id"] = W.argmax(axis=1) + 1
    assignments["topic_strength"] = W.max(axis=1)
    return pd.DataFrame(topic_rows), assignments

topic_summary_df, topic_assignments_df = discover_topics(n_topics=5)
if not topic_summary_df.empty:
    display(topic_summary_df)
    display(topic_assignments_df.head())

## 14. Compare papers by semantic similarity

In [ ]:
def build_paper_similarity():
    names = list(paper_names)
    vectors = []
    for name in names:
        indices = np.where(chunks_df["paper"].to_numpy() == name)[0]
        if len(indices):
            vectors.append(np.mean(chunk_embeddings[indices], axis=0))
        else:
            vectors.append(np.zeros(chunk_embeddings.shape[1], dtype=float))
    vectors = np.asarray(vectors)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    vectors = vectors / np.maximum(norms, 1e-12)
    matrix = vectors @ vectors.T
    return pd.DataFrame(matrix, index=names, columns=names)

paper_similarity_df = build_paper_similarity()
print("Paper-to-paper embedding similarity (higher means more similar in embedding space; not quality):")
display(paper_similarity_df.round(3))

if len(paper_names) >= 2:
    pairs = []
    for i in range(len(paper_names)):
        for j in range(i + 1, len(paper_names)):
            pairs.append({"paper_1": paper_names[i], "paper_2": paper_names[j],
                          "similarity": float(paper_similarity_df.iloc[i, j])})
    paper_pair_df = pd.DataFrame(pairs).sort_values("similarity", ascending=False).reset_index(drop=True)
    display(paper_pair_df.head(10))
else:
    paper_pair_df = pd.DataFrame(columns=["paper_1", "paper_2", "similarity"])
    print("Upload two or more papers to compare papers with one another.")

## 15. Mine DOIs and a reference-section preview

In [ ]:
DOI_RE = re.compile(r"\b10\.\d{4,9}/[-._;()/:A-Z0-9]+", re.IGNORECASE)

def mine_citations(paper_name: str) -> dict:
    text = "\n".join(pages_df.loc[pages_df.paper == paper_name, "text"].astype(str))
    dois = sorted(set(match.rstrip(".,;)") for match in DOI_RE.findall(text)))
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    reference_start = None
    for i, line in enumerate(lines):
        if re.match(r"^(references|bibliography|works cited)\s*$", line, flags=re.IGNORECASE):
            reference_start = i
            break
    references_preview = lines[reference_start + 1: reference_start + 21] if reference_start is not None else []
    return {"paper": paper_name, "dois_found": dois, "reference_section_found": reference_start is not None,
            "references_preview": references_preview}

citation_records = [mine_citations(paper) for paper in paper_names]
for record in citation_records:
    print(f"\n{record['paper']} — DOI-like identifiers: {len(record['dois_found'])}")
    print(record["dois_found"][:10])
    print("Reference section found:", record["reference_section_found"])
    if record["references_preview"]:
        print("Reference preview:", " | ".join(record["references_preview"][:5]))

## 16. Optional OCR for scanned PDFs

Use OCR only for pages where `PyMuPDF` found no selectable text. OCR is slower and may misread equations, tables, or symbols. This optional section installs Tesseract in the Colab runtime.

In [ ]:
# OPTIONAL setup — run these commands only when you have scanned/image-only PDFs.
# Uncomment and run both setup commands, then run the helper function below.
# !apt-get -qq update && apt-get -qq install -y tesseract-ocr
# %pip -q install pytesseract pillow

import io

def ocr_blank_pages(max_pages_per_paper=None, dpi=220):
    """OCR blank-text PDF pages, update pages_df, then rerun chunking/index cells.

    Requires the optional Tesseract and pytesseract setup above.
    OCR can misread equations, tables, multi-column layouts, and symbols; verify results.
    """
    import pytesseract
    from PIL import Image
    blank_rows = pages_df[pages_df["text"].fillna("").str.strip().eq("")]
    changed = 0
    for paper_name in blank_rows["paper"].drop_duplicates().tolist():
        rows = blank_rows[blank_rows["paper"] == paper_name]
        if max_pages_per_paper is not None:
            rows = rows.head(int(max_pages_per_paper))
        pdf_bytes = pdf_files[paper_name]
        with fitz.open(stream=pdf_bytes, filetype="pdf") as pdf:
            for row_index, row in rows.iterrows():
                page = pdf[int(row["page"]) - 1]
                pix = page.get_pixmap(dpi=dpi, alpha=False)
                image = Image.open(io.BytesIO(pix.tobytes("png")))
                recognized = clean_page_text(pytesseract.image_to_string(image))
                pages_df.loc[row_index, "text"] = recognized
                pages_df.loc[row_index, "word_count"] = len(recognized.split())
                changed += bool(recognized)
                print(f"OCR {paper_name} — page {int(row['page'])}: {len(recognized.split())} words")
    print(f"OCR produced selectable text on {changed} page(s). Now re-run the chunking and indexing cells below.")
    return pages_df

# After installing the optional dependencies, run:
# pages_df = ocr_blank_pages()
# Then rerun cells 5 onward (chunking, embeddings, retrieval, insights, and export).

## 17. Optional PDF table extraction

Tables in scientific PDFs can be difficult to recover perfectly. For PDFs that use clear grid/text layouts, `pdfplumber` is worth trying; verify every extracted table against the rendered page.

In [ ]:
# OPTIONAL setup and helper. Run only when you need to inspect tables.
# %pip -q install pdfplumber
# import io
# import pdfplumber
# def extract_pdf_tables(pdf_bytes: bytes):
#     found = []
#     with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
#         for page_number, page in enumerate(pdf.pages, start=1):
#             for table_number, table in enumerate(page.extract_tables() or [], start=1):
#                 found.append({"page": page_number, "table_number": table_number,
#                               "rows": table})
#     return found
#
# Example:
# table_results = extract_pdf_tables(next(iter(pdf_files.values())))
# print(f"Tables detected: {len(table_results)}")
# if table_results:
#     display(pd.DataFrame(table_results[0]["rows"]))

## 18. Retrieval evaluation scaffold — measure before claiming accuracy

Fill in relevant `chunk_id` values after manually reviewing the returned passages. The function reports Precision@K, Recall@K, and reciprocal rank for the queries you label. Empty ground truth is not evaluated.

In [ ]:
def evaluate_retrieval(annotations, k: int = 5, mode: str = "hybrid") -> pd.DataFrame:
    """annotations: list of {'query': ..., 'relevant_chunk_ids': [...]} reviewed by a human."""
    rows = []
    for item in annotations:
        query = str(item.get("query", "")).strip()
        relevant = set(item.get("relevant_chunk_ids", []))
        if not query or not relevant:
            continue
        retrieved = search_papers(query, top_k=k, mode=mode)["chunk_id"].tolist()
        hits = [1 if chunk_id in relevant else 0 for chunk_id in retrieved]
        precision = sum(hits) / max(1, len(retrieved))
        recall = sum(hits) / len(relevant)
        reciprocal_rank = next((1.0 / i for i, hit in enumerate(hits, start=1) if hit), 0.0)
        rows.append({"query": query, f"precision@{k}": precision,
                     f"recall@{k}": recall, "reciprocal_rank": reciprocal_rank,
                     "relevant_count": len(relevant), "retrieved_count": len(retrieved)})
    return pd.DataFrame(rows)

# Replace these placeholder IDs after manually checking search results; do not score unreviewed labels.
# reviewed_queries = [
#     {"query": "what are the limitations", "relevant_chunk_ids": ["PAPER.pdf::p3::w0"]},
#     {"query": "what dataset was used", "relevant_chunk_ids": ["PAPER.pdf::p5::w175"]},
# ]
# evaluation_df = evaluate_retrieval(reviewed_queries, k=5)
# display(evaluation_df)
# if not evaluation_df.empty:
#     display(evaluation_df.select_dtypes(include="number").mean().to_frame("mean").T)

## 19. Prepare paper briefs and export website-friendly insights

In [ ]:
def make_paper_brief(paper_name: str) -> dict:
    pages = pages_df[pages_df.paper == paper_name]
    meta_rows = papers_df[papers_df.paper == paper_name]
    text = " ".join(pages["text"].astype(str))
    summary = extractive_summary(paper_name, n_sentences=5)
    return {
        "paper": paper_name,
        "title": str(meta_rows.iloc[0]["title"]) if not meta_rows.empty else Path(paper_name).stem,
        "author": str(meta_rows.iloc[0]["author"]) if not meta_rows.empty else "",
        "pages": int(len(pages)),
        "words": int(sum(pages["word_count"].astype(int))),
        "doi_candidates": mine_citations(paper_name)["dois_found"],
        "top_keyphrases": [phrase for phrase, score in paper_keyword_map.get(paper_name, [])[:10]],
        "extractive_summary": [
            {"page": int(row.page), "sentence": str(row.sentence)}
            for row in summary.itertuples(index=False)
        ],
    }

paper_briefs = [make_paper_brief(paper) for paper in paper_names]
web_export = {
    "schema_version": "1.0",
    "project": "PaperLens Research Paper Intelligence Assistant",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "notice": "Summary metadata only. No full PDF text is exported by default.",
    "capabilities": ["semantic_search", "hybrid_retrieval", "optional_reranking",
                     "extractive_qa", "paper_similarity", "topic_discovery"],
    "papers": paper_briefs,
    "topic_summary": topic_summary_df.to_dict(orient="records") if not topic_summary_df.empty else [],
    "paper_similarity": paper_similarity_df.round(4).to_dict(),
}
print(f"Prepared lightweight metadata for {len(paper_briefs)} paper(s).")
display(pd.DataFrame([{
    "paper": item["paper"], "pages": item["pages"], "words": item["words"],
    "keywords": ", ".join(item["top_keyphrases"][:5])
} for item in paper_briefs]))

## 20. Export CSV, JSON, charts, and a ZIP package

In [ ]:
EXPORT_DIR = Path("/content/paperlens_advanced_exports")
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

pages_df.to_csv(EXPORT_DIR / "paper_pages.csv", index=False)
chunks_df.to_csv(EXPORT_DIR / "paper_chunks.csv", index=False)
papers_df.to_csv(EXPORT_DIR / "paper_metadata.csv", index=False)
paper_similarity_df.to_csv(EXPORT_DIR / "paper_similarity_matrix.csv")
if not paper_pair_df.empty:
    paper_pair_df.to_csv(EXPORT_DIR / "paper_similarity_pairs.csv", index=False)
if not topic_summary_df.empty:
    topic_summary_df.to_csv(EXPORT_DIR / "topic_summary.csv", index=False)
    topic_assignments_df.to_csv(EXPORT_DIR / "topic_assignments.csv", index=False)

with open(EXPORT_DIR / "paperlens_web_export.json", "w", encoding="utf-8") as f:
    json.dump(web_export, f, ensure_ascii=False, indent=2)
with open(EXPORT_DIR / "paper_briefs.json", "w", encoding="utf-8") as f:
    json.dump(paper_briefs, f, ensure_ascii=False, indent=2)

# Create a per-page word-count chart as a reproducible artifact.
fig, ax = plt.subplots(figsize=(10, 4))
for paper in pages_df["paper"].drop_duplicates():
    subset = pages_df[pages_df.paper == paper]
    ax.plot(subset["page"], subset["word_count"], marker="o", linewidth=1.5, label=Path(paper).stem[:24])
ax.set_title("Extracted words by PDF page")
ax.set_xlabel("PDF page")
ax.set_ylabel("Word count")
ax.grid(True, alpha=0.25)
if pages_df["paper"].nunique() <= 8:
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(EXPORT_DIR / "words_per_page.png", dpi=160)
plt.show()
plt.close(fig)

# Write a clear README for the exported assets.
(EXPORT_DIR / "README.txt").write_text(
    "PaperLens Advanced Colab export\n\n"
    "paper_pages.csv and paper_chunks.csv contain extracted PDF text; keep them private if the PDFs are private/copyrighted.\n"
    "paperlens_web_export.json contains summary metadata and insights, not the full document text.\n"
    "Never publish extracted text, unpublished manuscripts, or restricted PDFs without permission.\n",
    encoding="utf-8"
)

zip_path = Path("/content/paperlens_advanced_exports.zip")
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in EXPORT_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=f"paperlens_advanced_exports/{file_path.relative_to(EXPORT_DIR)}")

print("Created:", zip_path)
print("Export files:")
for path in sorted(EXPORT_DIR.iterdir()):
    print(f" • {path.name} ({path.stat().st_size:,} bytes)")
# Click the download prompt below to save the archive to your computer.
files.download(str(zip_path))

## 21. Advanced roadmap — what this notebook does and does not do

**Implemented in the notebook:** semantic embeddings, hybrid RRF retrieval, optional cross-encoder reranking, extractive QA with page-level evidence, extractive summaries, optional transformer generation, keyphrases, NMF topics, paper similarity, DOI/reference mining, optional OCR and table extraction, retrieval-metric scaffold, and exports.

**Next production upgrades (require additional backend engineering):** a FastAPI endpoint for secure multi-user sessions; a persistent vector database such as FAISS/Qdrant/Chroma; authentication and request limits; a worker queue for large PDFs; proper observability; explicit user consent and retention controls; automated regression tests; and an LLM provider integration managed by server-side secrets. Do not put secret API keys in GitHub Pages JavaScript or a public Colab notebook.

**Website architecture note:** GitHub Pages serves static HTML/CSS/JavaScript only. The website's browser demo can process local PDFs and may offer in-browser semantic features, but it cannot execute this full Python notebook or securely hold a provider API key. For the full Python backend, deploy Streamlit or a separate API service.